# Breast Cancer CITE-seq Dataset

Build the paired gene and protein AnnData objects, align shared barcodes, and save the final combined dataset.

In [1]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
protein_root = Path("/Data/CITE")
gene_root = Path("/Data/GSE176078")
# output_path = Path("/Data/breast_cancer_cite.h5ad")


def normalize_barcode(barcode: str) -> str:
    return barcode.split("_", 1)[1] if "_" in barcode else barcode


def load_protein_sample(sample_dir: Path) -> ad.AnnData:
    adata = sc.read_mtx(sample_dir / "umi_count" / "matrix.mtx").transpose()
    adata.var_names = pd.read_csv(sample_dir / "umi_count" / "features.tsv", header=None)[0].astype(str).to_numpy()
    adata.obs_names = pd.read_csv(sample_dir / "umi_count" / "barcodes.tsv", header=None)[0].astype(str).to_numpy()
    return adata


def load_gene_sample(sample_dir: Path) -> ad.AnnData:
    adata = sc.read_mtx(sample_dir / "count_matrix_sparse.mtx").transpose()
    adata.var_names = pd.read_csv(sample_dir / "count_matrix_genes.tsv", header=None)[0].astype(str).to_numpy()
    adata.obs_names = (
        pd.read_csv(sample_dir / "count_matrix_barcodes.tsv", header=None)[0]
        .astype(str)
        .map(normalize_barcode)
        .to_numpy()
    )

    metadata = pd.read_csv(sample_dir / "metadata.csv")
    adata.obs["celltype_minor"] = metadata["celltype_minor"].to_numpy()
    adata.obs["celltype_major"] = metadata["celltype_major"].to_numpy()
    return adata

In [ ]:
protein_sample_dirs = {
    path.name.split("_", 1)[0]: path
    for path in protein_root.iterdir()
    if path.is_dir()
}
gene_sample_dirs = {path.name[3:]: path for path in gene_root.iterdir() if path.is_dir()}
sample_ids = sorted(set(protein_sample_dirs) & set(gene_sample_dirs))

if not sample_ids:
    raise ValueError("No matching CITE and gene samples were found.")

paired_adatas = []
protein_var_names = None

for sample_id in sample_ids:
    protein_adata = load_protein_sample(protein_sample_dirs[sample_id])
    gene_adata = load_gene_sample(gene_sample_dirs[sample_id])

    shared_barcodes = protein_adata.obs_names.intersection(gene_adata.obs_names)
    protein_adata = protein_adata[shared_barcodes].copy()
    gene_adata = gene_adata[shared_barcodes].copy()

    if protein_var_names is None:
        protein_var_names = protein_adata.var_names.to_numpy()
    elif not np.array_equal(protein_var_names, protein_adata.var_names.to_numpy()):
        raise ValueError(f"Protein features do not match for sample {sample_id}.")

    gene_adata.obsm["protein_counts"] = protein_adata.X.copy()
    paired_adatas.append(gene_adata)

    print(f"{sample_id}: {gene_adata.n_obs} cells, {gene_adata.n_vars} genes, {protein_adata.n_vars} proteins")

if protein_var_names is None:
    raise RuntimeError("Protein feature names were not loaded.")

adata = ad.concat(paired_adatas, join="outer", label="batch", keys=sample_ids, index_unique="-")
adata.uns["protein_var_names"] = protein_var_names
adata.obsm['protein_counts'] = pd.DataFrame(adata.obsm['protein_counts'].todense(), index=adata.obs_names, columns=adata.uns['protein_var_names'])
print(adata)

3838: 2316 cells, 29733 genes, 118 proteins
3946: 696 cells, 29733 genes, 118 proteins
4040: 2521 cells, 29733 genes, 118 proteins
4515: 1837 cells, 29733 genes, 118 proteins
AnnData object with n_obs × n_vars = 7370 × 29733
    obs: 'celltype_minor', 'celltype_major', 'batch'
    uns: 'protein_var_names'
    obsm: 'protein_counts'


In [ ]:
adata.write(output_path)
print(f"Saved to {output_path}")